#Setting up Drive and Dagshub/MLflow

In [1]:

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

!pip install -q prophet

import pandas as pd
import numpy as np

Mounted at /content/drive


In [2]:
!pip install -q dagshub mlflow

import dagshub
import mlflow

from mlflow import MlflowClient
from mlflow.entities import ViewType

dagshub.init(
    repo_owner="skupr23",
    repo_name="ml_final",
    mlflow=True,
)

EXPERIMENT_NAME = "Prophet_Training"


if mlflow.active_run() is not None:
    mlflow.end_run()


client = MlflowClient()

matches = client.search_experiments(
    view_type=ViewType.ALL,
    filter_string=f"name = '{EXPERIMENT_NAME}'",
)

if matches:
    experiment = matches[0]

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)
        print(
            f"Restored deleted experiment: "
            f"{EXPERIMENT_NAME} ({experiment.experiment_id})"
        )

mlflow.set_experiment(EXPERIMENT_NAME)

active_experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Experiment   :", active_experiment.name)
print("Experiment ID:", active_experiment.experiment_id)
print("Lifecycle    :", active_experiment.lifecycle_stage)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2549c785-86b8-423a-ac00-ffea31a9d7f4&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d0f39eb07a785ce1f747fd0ed842b85ab0a12dd194858a2a0029370a17c5a8d6




Accessing as skupr23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI : https://dagshub.com/skupr23/ml_final.mlflow
Experiment   : Prophet_Training
Experiment ID: 8
Lifecycle    : active


# Cell 2 - Import model

In [3]:

from prophet import Prophet
print('Prophet imported OK')

Prophet imported OK


# Cell 3 - Load processed data

In [4]:

train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


# Cell 4 - WMAE metric (same scoring function used everywhere)

In [5]:

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Cell 5 - Holiday dates handled

In [6]:

superbowl = ['2010-02-12','2011-02-11','2012-02-10','2013-02-08']
laborday  = ['2010-09-10','2011-09-09','2012-09-07','2013-09-06']
thanksgiving = ['2010-11-26','2011-11-25','2012-11-23','2013-11-29']
christmas = ['2010-12-31','2011-12-30','2012-12-28','2013-12-27']

holidays_df = pd.concat([
    pd.DataFrame({'holiday': 'SuperBowl', 'ds': pd.to_datetime(superbowl)}),
    pd.DataFrame({'holiday': 'LaborDay', 'ds': pd.to_datetime(laborday)}),
    pd.DataFrame({'holiday': 'Thanksgiving', 'ds': pd.to_datetime(thanksgiving)}),
    pd.DataFrame({'holiday': 'Christmas', 'ds': pd.to_datetime(christmas)}),
])
holidays_df['lower_window'] = -1
holidays_df['upper_window'] = 1
holidays_df.head()

,holiday,ds,lower_window,upper_window
0,SuperBowl,2010-02-12,-1,1
1,SuperBowl,2011-02-11,-1,1
2,SuperBowl,2012-02-10,-1,1
3,SuperBowl,2013-02-08,-1,1
0,LaborDay,2010-09-10,-1,1


---
# Cell 6 - Fit and score Prophet on the same population as XGBoost / LightGBM



In [7]:

import time
import logging

from prophet.serialize import model_to_json

logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

VAL_WEEKS = 39
MAX_SERIES = None
MIN_FIT_WEEKS = 2
KEEP_MODELS = True
PROGRESS_EVERY = 250

PROPHET_KWARGS = dict(
    holidays=holidays_df,
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='additive',
)

_t = train.sort_values('Date')
VAL_START = _t['Date'].max() - pd.Timedelta(weeks=VAL_WEEKS)
gbm_val = _t[_t['Date'] >= VAL_START].copy()
gbm_fit = _t[_t['Date'] < VAL_START].copy()

print('fit ends:  ', gbm_fit['Date'].max())
print('val range: ', gbm_val['Date'].min(), 'to', gbm_val['Date'].max())
print('val rows:  ', len(gbm_val),
      '| val dates:', gbm_val['Date'].nunique(),
      '| val pairs:', gbm_val[['Store', 'Dept']].drop_duplicates().shape[0])

baseline_pred = gbm_val['lag_52'].fillna(gbm_val['storedept_median_sales'])
baseline_wmae = wmae(gbm_val['Weekly_Sales'], baseline_pred, gbm_val['IsHoliday'])
print('\n52-week seasonal baseline WMAE:', baseline_wmae)
print('   XGBoost / LightGBM reported:  1789.9133525504185')
if MAX_SERIES is None and abs(baseline_wmae - 1789.9133525504185) > 1e-6:
    raise AssertionError('population mismatch: this is not the GBM validation set')

sd_median = gbm_fit.groupby(['Store', 'Dept'])['Weekly_Sales'].median().to_dict()
d_median = gbm_fit.groupby('Dept')['Weekly_Sales'].median().to_dict()
g_median = float(gbm_fit['Weekly_Sales'].median())

fit_groups = {k: g for k, g in gbm_fit.groupby(['Store', 'Dept'], sort=False)}

gbm_val = gbm_val.sort_values(['Store', 'Dept', 'Date'])
pair_groups = list(gbm_val.groupby(['Store', 'Dept'], sort=False))
if MAX_SERIES is not None:
    pair_groups = pair_groups[:MAX_SERIES]

prophet_models = {}
pred_parts = []
n_prophet = n_fallback = n_failed = 0
start = time.time()

for i, ((store, dept), vg) in enumerate(pair_groups, 1):
    fg = fit_groups.get((store, dept))
    use_fallback = fg is None or len(fg) < MIN_FIT_WEEKS

    if not use_fallback:
        try:
            model = Prophet(**PROPHET_KWARGS)
            model.fit(fg[['Date', 'Weekly_Sales']].rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'}))

            yhat = model.predict(pd.DataFrame({'ds': vg['Date'].to_numpy()}))['yhat'].to_numpy()
            if KEEP_MODELS:
                prophet_models[(store, dept)] = model_to_json(model)
            n_prophet += 1
        except Exception:
            use_fallback = True
            n_failed += 1

    if use_fallback:
        v = sd_median.get((store, dept), np.nan)
        if not np.isfinite(v):
            v = d_median.get(dept, np.nan)
        if not np.isfinite(v):
            v = g_median
        yhat = np.full(len(vg), float(v))
        n_fallback += 1

    pred_parts.append(pd.Series(yhat, index=vg.index))

    if i % PROGRESS_EVERY == 0:
        print(f'  {i}/{len(pair_groups)} pairs | {time.time()-start:.0f}s elapsed')

elapsed = time.time() - start

all_pred_s = pd.concat(pred_parts)
scored = gbm_val.loc[all_pred_s.index]
all_pred = all_pred_s.to_numpy()
all_true = scored['Weekly_Sales'].to_numpy()
all_holiday = scored['IsHoliday'].to_numpy().astype(bool)

prophet_wmae = wmae(all_true, all_pred, all_holiday)
prophet_wmae_holiday = wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday])
prophet_wmae_non_holiday = wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday])

_s = scored.assign(pred=all_pred)
_s['_w'] = np.where(all_holiday, 5, 1)
_s['_wae'] = _s['_w'] * (_s['Weekly_Sales'] - _s['pred']).abs()
per_pair = (
    _s.groupby(['Store', 'Dept'])
      .agg(_wae_sum=('_wae', 'sum'),
           _w_sum=('_w', 'sum'),
           mean_weekly_sales=('Weekly_Sales', 'mean'),
           n_val_weeks=('Weekly_Sales', 'size'))
      .reset_index()
)
per_pair['wmae'] = per_pair['_wae_sum'] / per_pair['_w_sum']
per_pair = (per_pair[['Store', 'Dept', 'wmae', 'mean_weekly_sales', 'n_val_weeks']]
            .sort_values('wmae')
            .reset_index(drop=True))

population_complete = len(scored) == len(gbm_val)

print(f'\nFitted {n_prophet} Prophet models, {n_fallback} median fallbacks '
      f'({n_failed} of them after a fit error), in {elapsed:.0f}s')
print('Rows scored:', len(scored), '| full population:', population_complete)
print('\nBaseline (52wk):          ', baseline_wmae)
print('Prophet (full population):', prophet_wmae)
print('  XGBoost:                ', 1281.080964551936)
print('  LightGBM:               ', 1297.4414789901796)
print('\nHoliday WMAE:    ', prophet_wmae_holiday)
print('Non-holiday WMAE:', prophet_wmae_non_holiday)

fit ends:   2012-01-20 00:00:00
val range:  2012-01-27 00:00:00 to 2012-10-26 00:00:00
val rows:   118540 | val dates: 40 | val pairs: 3208

52-week seasonal baseline WMAE: 1789.9133525504185
   XGBoost / LightGBM reported:  1789.9133525504185
  250/3208 pairs | 79s elapsed
  500/3208 pairs | 163s elapsed
  750/3208 pairs | 215s elapsed
  1000/3208 pairs | 332s elapsed
  1250/3208 pairs | 383s elapsed
  1500/3208 pairs | 426s elapsed
  1750/3208 pairs | 491s elapsed
  2000/3208 pairs | 606s elapsed
  2250/3208 pairs | 695s elapsed
  2500/3208 pairs | 746s elapsed
  2750/3208 pairs | 839s elapsed
  3000/3208 pairs | 909s elapsed

Fitted 3167 Prophet models, 41 median fallbacks (0 of them after a fit error), in 1007s
Rows scored: 118540 | full population: True

Baseline (52wk):           1789.9133525504185
Prophet (full population): 1905.4227245923344
  XGBoost:                 1281.080964551936
  LightGBM:                1297.4414789901796

Holiday WMAE:     1896.9936330616104
Non-holid

# Cell 7 - Save Models locally

In [8]:

import os
import joblib

os.makedirs('models', exist_ok=True)

joblib.dump(prophet_models, 'models/prophet_best.pkl')

size_mb = os.path.getsize('models/prophet_best.pkl') / 1e6
print(f'Saved {len(prophet_models)} models ({size_mb:.1f} MB).')

Saved 3167 models (79.9 MB).


# Cell 8 - MLflow tracking

In [9]:

import os
import tempfile

PREPROCESSING_STEPS = [
    {'step': 'load_processed', 'detail': 'read data/train_processed.pkl, sort by Store-Dept-Date'},
    {'step': 'chronological_split',
     'detail': f"val = rows with Date >= max(Date) - {VAL_WEEKS} weeks; identical to the XGBoost/LightGBM notebooks"},
    {'step': 'no_series_filter', 'detail': 'every Store-Dept pair in the window is scored; no minimum-length filter'},
    {'step': 'no_resampling', 'detail': 'no asfreq / interpolation: every scored row is a real observation'},
    {'step': 'holiday_regressors',
     'detail': 'the four competition holidays passed to Prophet as an explicit holidays frame, +/-1 week window'},
    {'step': 'per_pair_fit', 'detail': "one Prophet per Store-Dept pair, fitted on that pair's history only"},
    {'step': 'fallback_aggregates',
     'detail': 'fit-portion median (store-dept -> dept -> global) for pairs with too little history'},
]


with mlflow.start_run(run_name='Prophet_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'val_weeks': VAL_WEEKS,
        'series_filter': 'none (all Store-Dept pairs in the validation window)',
        'resampling': 'none (no asfreq, no interpolation)',
        'merge_validate': 'many_to_one',
        'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })
    mlflow.log_metrics({
        'train_rows': train.shape[0],
        'train_cols': train.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()),
        'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store', 'Dept']].drop_duplicates().shape[0]),
        'fit_rows': len(gbm_fit),
        'val_rows': len(gbm_val),
        'val_dates': int(gbm_val['Date'].nunique()),
        'val_pairs': int(gbm_val[['Store', 'Dept']].drop_duplicates().shape[0]),
    })
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )

with mlflow.start_run(run_name='Prophet_Fitting'):
    mlflow.set_tag('stage', 'fitting')
    mlflow.log_params({
        'yearly_seasonality': True,
        'weekly_seasonality': False,
        'daily_seasonality': False,
        'seasonality_mode': 'additive',
        'holidays': ['SuperBowl', 'LaborDay', 'Thanksgiving', 'Christmas'],
        'holiday_lower_window': -1,
        'holiday_upper_window': 1,
        'holiday_source': 'competition dates from feature_engineering.ipynb, not Prophet US calendar',
        'min_fit_weeks': MIN_FIT_WEEKS,
        'fallback': 'fit-portion median (store-dept -> dept -> global)',
    })
    mlflow.log_metrics({
        'n_holiday_rows': len(holidays_df),
        'n_prophet_models': n_prophet,
        'n_median_fallbacks': n_fallback,
        'n_fit_failures': n_failed,
        'fit_seconds': elapsed,
        'per_pair_wmae_mean': float(per_pair['wmae'].mean()),
        'per_pair_wmae_median': float(per_pair['wmae'].median()),
        'per_pair_wmae_min': float(per_pair['wmae'].min()),
        'per_pair_wmae_max': float(per_pair['wmae'].max()),

        'per_pair_wmae_vs_volume_corr': float(per_pair['wmae'].corr(per_pair['mean_weekly_sales'])),
    })
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'per_pair_wmae.csv')
        per_pair.to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='fitting')

with mlflow.start_run(run_name='Prophet_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.set_tag('population', 'full' if population_complete else 'partial')
    mlflow.log_params({
        'scheme': 'chronological holdout (no k-fold: the target is a time series), rebuilt exactly as in the XGBoost/LightGBM notebooks',
        'val_weeks': VAL_WEEKS,
        'val_start': str(gbm_val['Date'].min().date()),
        'val_end': str(gbm_val['Date'].max().date()),
        'forecast_strategy': "one Prophet per Store-Dept pair, predicted on that pair's real validation dates",
        'max_series': str(MAX_SERIES),
        'population_complete': population_complete,
        'scored_on': 'same rows as XGBoost/LightGBM: directly comparable',
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_wmae,
        'final_wmae': prophet_wmae,
        'val_wmae_holiday': prophet_wmae_holiday,
        'val_wmae_non_holiday': prophet_wmae_non_holiday,
        'improvement_over_baseline': baseline_wmae - prophet_wmae,
        'val_rows': len(scored),
        'val_pairs': len(pair_groups),
    })

with mlflow.start_run(run_name='Prophet_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'prophet'})
    mlflow.log_params({
        'model_type': 'local, one Prophet model per Store-Dept pair',
        'n_models': len(prophet_models),
        'seasonality_mode': 'additive',
        'val_weeks': VAL_WEEKS,
        'serialization': 'prophet.serialize.model_to_json, dict keyed by (Store, Dept)',
    })
    mlflow.log_metrics({
        'final_wmae': prophet_wmae,
        'baseline_wmae': baseline_wmae,
        'per_pair_wmae_median': float(per_pair['wmae'].median()),
    })

    mlflow.log_artifact('models/prophet_best.pkl', artifact_path='estimator')
    print('Prophet final run:', final_run.info.run_id)

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run Prophet_Cleaning at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8/runs/c99a462bea254b849ca456122151b685
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8
🏃 View run Prophet_Fitting at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8/runs/849707f4d308474b93dd09bfa825537f
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8
🏃 View run Prophet_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8/runs/7aa9e05110ec4aa9981a9118139d5896
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8
Prophet final run: 186b0dd575e24921ac770537586f4063
🏃 View run Prophet_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8/runs/186b0dd575e24921ac770537586f4063
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8
